# Experiment 06: Enhancing Retrieval Accuracy via Query Transformation
- **Goal:** Improve the hit rate of vector similarity searches by translating ambiguous user queries into precise domain terminology using a keyword dictionary.
- **Components:** `ChatPromptTemplate`, `dictionary_chain`, `PineconeVectorStore`, `LCEL`.
- **Method:** 
    1. **Dictionary Definition:** Create a mapping of colloquial terms to professional legal/tax terminology (e.g., "Office Worker" -> "Resident").
    2. **Query Transformation:** Implement a pre-processing LLM chain that rewrites the user's question based on the provided dictionary.
    3. **Integrated Pipeline:** Combine the transformation logic with the RAG chain so that the retriever always receives optimized search queries.
- **Key Observation:** Query transformation ensures consistency between user intent and indexed document terminology. It prevents retrieval failure even when users use non-technical language.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

load_dotenv()
embedding = OpenAIEmbeddings(model='text-embedding-3-large')

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# 기존에 생성된 Pinecone 인덱스(tax-index)에 연결
index_name = "tax-index"
database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langsmith import Client

retriever = database.as_retriever()

client = Client()
rag_prompt = client.pull_prompt("rlm/rag-prompt")

llm = ChatOpenAI(model='gpt-4o')

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 그대로 반환해주세요.
    사전: {dictionary}

    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()

In [4]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
new_question = dictionary_chain.invoke({"question": query})
print(f'변환된 질문: {new_question}')

변환된 질문: 연봉 5천만원인 거주자의 소득세는 얼마인가요?


In [5]:
tax_chain = dictionary_chain | rag_chain
ai_response = tax_chain.invoke({"question": query})
print(f'AI 응답: {ai_response}')

AI 응답: 연봉 5천만원인 거주자의 소득세는 624만원입니다. 이 금액은 소득세율표에서 5천만원까지의 구간에 해당하는 84만원에 추가로 1,400만원을 초과하는 금액의 15퍼센트를 더해 계산됩니다.


In [6]:
query = '연봉 5천만원인 거주자의 종합소득세는 얼마인가요?'
new_question = dictionary_chain.invoke({"question": query})
print(f'변환된 질문: {new_question}')

변환된 질문: 연봉 5천만원인 거주자의 종합소득세는 얼마인가요?


In [7]:
tax_chain = dictionary_chain | rag_chain
ai_response = tax_chain.invoke({"question": query})
print(f'AI 응답: {ai_response}')

AI 응답: 연봉 5천만 원인 거주자의 종합소득세는 624만 원입니다. 이는 과세표준이 5,000만 원에 해당하며, 계산은 624만 원의 고정 금액을 적용하게 됩니다.
